## Data Migration: SQL to Postgres

In [3]:
import os
import pandas as pd
import pyodbc
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv


## 1. LOAD CREDENTIALS

In [4]:
load_dotenv()

True

In [5]:
sql_host = os.getenv("SQLSERVER_HOST")
sql_db = os.getenv("SQLSERVER_DB")
sql_user = os.getenv("SQLSERVER_USER")
sql_password = os.getenv("SQLSERVER_PASSWORD")


In [6]:
print(f"SQL SERVER HOST:{sql_host}") 
print(f"SQL DB:{sql_db}") 
print(f"SQL USER:{sql_user}") 
print(f"SQL PASSWORD:{sql_password}") 

SQL SERVER HOST:DESKTOP-P6OS4A2\SQLEXPRESS
SQL DB:AdventureWorks2022
SQL USER:loadingusr
SQL PASSWORD:Toluca2000


In [7]:
pg_host= os.getenv("PostgreSQL_HOST")
pg_port= os.getenv("PostgreSQL_PORT")
pg_db= os.getenv("PostgreSQL_DB")
pg_user= os.getenv("PostgreSQL_USER")
pg_password= os.getenv("PostgreSQL_PASSWORD")

In [8]:
print(f"POSTGRE HOST:{pg_host}")
print(f"POSTGRE PORT:{pg_port}")
print(f"POSTGRE DB:{pg_db}")
print(f"POSTGRE USER:{pg_user}")
print(f"POSTGRE PASSWORD:{pg_password}")

POSTGRE HOST:localhost
POSTGRE PORT:5432
POSTGRE DB:dvdrental
POSTGRE USER:loadingusr
POSTGRE PASSWORD:Toluca2000


## Connect to SQL Server

In [9]:
print("connect to SQL Server...")
print(f"    Server:{sql_host}")
print(f"    User:{sql_user}")
print(f"    Password:{sql_password}")
print(f"    DB:{sql_db}")

connect to SQL Server...
    Server:DESKTOP-P6OS4A2\SQLEXPRESS
    User:loadingusr
    Password:Toluca2000
    DB:AdventureWorks2022


In [15]:
print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server']


In [ ]:
try:
    sql_conn_string =(
        f"USER={sql_user};"
        f"PASSWORD={sql_password};"
        f"SERVER={sql_host};"
        f"DATABASE={sql_db};"
        f"DRIVER={{SQL Server}};"
        f"Trusted_Conection=yes;"
    )
    sql_conn = pyodbc.connect(sql_conn_string)
    sql_cursor = sql_conn.cursor()
    print(f"[SUCCESS]Conection to SQL Server completed!")
    

except Exception as e:
    print(f"SQL Server connection failed:{e}")
    print(""" How to Toubleshoot:
         >1. Check server name in .env is correct
         >2. Verify SQL Server is runing
         >3. Check Windows authentication is enable
          ... 
""")

SQL Server conection successful!


## Connect to PostgreSQL

In [10]:
print("connect to SQL PostgreSQL...")
print(f"    Server:{pg_host}")
print(f"    User:{pg_db}")


connect to SQL PostgreSQL...
    Server:localhost
    User:dvdrental


In [11]:
try:
    pg_conn = psycopg2.connect(
        host=pg_host,
        port=pg_port,
        database=pg_db,
        user=pg_user,
        password=pg_password
    )

    pg_cursor=pg_conn.cursor()
    pg_cursor.execute("SELECT version();")
    pg_version = pg_cursor.fetchone()[0]
    print("Conected to PostgreSQL")
    print(f"    Version: {pg_version[:50]}...\n")

except psycopg2.OperationalError as e:
    print(f"Postgres Connection failed{e}")
    print(""" How to trobleshoot:
            > 1. Check Postgres is running 
            > 2. Verify user name + password
            > 3. Check db exists 
          
          ...
""")
    
except Exception as e:
    print(f"Unexpected error: {e}")
    raise

Conected to PostgreSQL
    Version: PostgreSQL 18.3 on x86_64-windows, compiled by msv...



## 4. Define the tables to migrate

### Migration Orden
- Sales 
- Production
- Purchasing

In [11]:
tables_to_migrate= ['Sales', 'Production', 'Purchasing']
print(tables_to_migrate)

['Sales', 'Production', 'Purchasing']


In [13]:
print("Tables to migrate:")
for i, table in enumerate(tables_to_migrate, 1):
    print(f"    {i}.    {table}")

total_no_tbls = len(tables_to_migrate)
print(f"\nTotal number of tables to migrate: {total_no_tbls}")
    

Tables to migrate:
    1.    Sales
    2.    Production
    3.    Purchasing

Total number of tables to migrate: 3


## 5. Run pre migration checks

In [14]:
print("==" * 50)
print(">> Rows counts")
print("==" * 50)

>> Rows counts


In [15]:
test_query = "SELECT COUNT(*) AS total_rows FROM customer;"
pg_cursor.execute(test_query)
count = pg_cursor.fetchone()[0]

print(f"Result: {count}")

Result: 599


In [ ]:
base_line_counts = {}